In [1]:
%pip install openai langchain langchain-community faiss-cpu

  Using cached langchain_community-0.3.27-py3-none-any.whl.metadata (2.9 kB)
  Using cached distro-1.9.0-py3-none-any.whl.metadata (6.8 kB)
  Using cached async_timeout-4.0.3-py3-none-any.whl.metadata (4.2 kB)
  Using cached jsonpatch-1.33-py2.py3-none-any.whl.metadata (3.0 kB)
  Using cached jsonpointer-3.0.0-py2.py3-none-any.whl.metadata (2.3 kB)
  Using cached greenlet-3.2.3-cp310-cp310-win_amd64.whl.metadata (4.2 kB)
  Using cached dataclasses_json-0.6.7-py3-none-any.whl.metadata (25 kB)
  Using cached pydantic_settings-2.10.1-py3-none-any.whl.metadata (3.4 kB)
  Using cached httpx_sse-0.4.1-py3-none-any.whl.metadata (9.4 kB)
  Using cached aiosignal-1.4.0-py3-none-any.whl.metadata (3.7 kB)
  Using cached marshmallow-3.26.1-py3-none-any.whl.metadata (7.3 kB)
  Using cached typing_inspect-0.9.0-py3-none-any.whl.metadata (1.5 kB)
  Using cached python_dotenv-1.1.1-py3-none-any.whl.metadata (24 kB)
  Using cached mypy_extensions-1.1.0-py3-none-any.whl.metadata (1.1 kB)
  Using cached 

In [5]:
%pip install sentence-transformers

Note: you may need to restart the kernel to use updated packages.


In [3]:
from getpass import getpass
import os

# 🔑 Set Groq API Key & Endpoint
os.environ["OPENAI_API_KEY"] = getpass("Enter your Token: ")
os.environ["OPENAI_API_BASE"] = "https://api.groq.com/openai/v1"

In [6]:
import json
import os
from langchain.chat_models import ChatOpenAI
from langchain.memory import ConversationSummaryMemory, VectorStoreRetrieverMemory
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain.schema import AIMessage, HumanMessage

# --- 1. GROQ SETUP --- os.environ["OPENAI_API_KEY"] = "your_groq_api_key" #
# 🔐 Replace with your Groq API key
# os.environ["OPENAI_API_BASE"] = "https://api.groq.com/openai/v1"

llm = ChatOpenAI(
    model="llama3-70b-8192",  # ✅ or 'llama3-70b-8192'
    temperature=0.3
)

# --- 2. MEMORY SETUP ---
embedding = HuggingFaceEmbeddings()

chat_memory = ConversationSummaryMemory(
    llm=llm,
    memory_key="chat_history",
    return_messages=True
)

vector_store = FAISS.from_texts(["example event"], embedding)
event_memory = VectorStoreRetrieverMemory(
    retriever=vector_store.as_retriever(search_kwargs={"k": 3}),
    memory_key="events"
)

# --- 3. SUMMARIZER ---
def summarize_events(events):
    if not events:
        return "No events detected."
    flat = "\n".join(f"{e['timestamp']}: {e['event_type']} at {e['location']} (confidence: {e['confidence']})" for e in events)
    prompt = f"Summarize the following surveillance events clearly and concisely:\n\n{flat}"
    return llm.invoke(prompt)

# --- 4. CHAT HANDLER ---
def handle_chat(user_input, summary_text):
    event_memory.save_context({"input": "event_summary"}, {"output": summary_text})
    chat_memory.chat_memory.add_user_message(user_input)

    retrieved = event_memory.load_memory_variables({"input": user_input})
    chat_history = chat_memory.load_memory_variables({"input": user_input}).get("chat_history", [])

    formatted = "\n".join([
        f"User: {msg.content}" if isinstance(msg, HumanMessage) else f"Assistant: {msg.content}"
        for msg in chat_history
    ])

    prompt = f"""
You are an intelligent assistant helping summarize and analyze security surveillance data.

Context:
{retrieved.get("events", "")}

Conversation History:
{formatted}

User: {user_input}
Assistant:"""

    response = llm.invoke(prompt)
    chat_memory.chat_memory.add_ai_message(response)
    return response



C:\Users\prans\AppData\Local\Temp\ipykernel_3808\1416018144.py:19: LangChainDeprecationWarning: Default values for HuggingFaceEmbeddings.model_name were deprecated in LangChain 0.2.16 and will be removed in 0.4.0. Explicitly pass a model_name to the HuggingFaceEmbeddings constructor instead.
  embedding = HuggingFaceEmbeddings()
d:\Hackhathon\Vuencode Hackathon\video processing\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\prans\AppData\Local\Temp\ipykernel_3808\1416018144.py:21: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  chat_memory = ConversationSummaryMemory(
C:\Users\prans\AppData\Local\Temp\ipykernel_3808\1416018144.py:28: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain

In [10]:
if __name__ == "__main__":
    # load the events from a JSON file
    events_file = "detected_violations.json"
    if not os.path.exists(events_file):
        print(f"Error: {events_file} not found. Please create a JSON file with surveillance events.")
        exit(1)
    with open(events_file, "r") as f:
        try:
            events = json.load(f)
        except json.JSONDecodeError as e:
            print(f"Error reading {events_file}: {e}")
            exit(1)

    print("🔎 Generating Summary...")
    summary = summarize_events(events)
    print("📄 Summary:\n", summary)

    print("\n💬 Ask a question about the events:")
    while True:
        user_input = input("You: ")
        if user_input.lower() in ["exit", "quit"]:
            break
        reply = handle_chat(user_input, summary)
        print("Assistant:", reply.content)

🔎 Generating Summary...
📄 Summary:
 content="Here is a clear and concise summary of the surveillance events:\n\n**Zone A**\n\n* **No Helmet**: Detected 14 times between 21:35:15 and 21:35:29 with a confidence level of 0.85.\n* **No Seatbelt**: Detected 24 times between 21:35:15 and 21:35:29 with a confidence level of 0.8.\n* **Mobile Usage While Driving**: Detected 21 times between 21:35:15 and 21:35:29 with a confidence level of 0.75.\n\nNote: The events are listed in the order they occurred, but I've grouped them by type to make it easier to read." additional_kwargs={} response_metadata={'token_usage': {'completion_tokens': 145, 'prompt_tokens': 2845, 'total_tokens': 2990, 'completion_tokens_details': None, 'prompt_tokens_details': None, 'queue_time': 0.279384933, 'prompt_time': 0.107813283, 'completion_time': 0.603632656, 'total_time': 0.711445939}, 'model_name': 'llama3-70b-8192', 'system_fingerprint': 'fp_bf16903a67', 'finish_reason': 'stop', 'logprobs': None} id='run--62417e5c-d4